In [1]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.9 MB/s eta 0:00:00


In [2]:
!pip install -q langchain-huggingface transformers accelerate

In [3]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
from google.colab import userdata

token = userdata.get('HF_TOKEN')

In [5]:
huggingfacehub_api_token=token

In [6]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/cb5882416d47e594e5daa73b/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [7]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [8]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1780185601,
 'time_last_update_utc': 'Sun, 31 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1780272001,
 'time_next_update_utc': 'Mon, 01 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.2043}

In [9]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [28]:
from langchain_huggingface import ChatHuggingFace
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="microsoft/Phi-4-mini-instruct",
    max_new_tokens=200,
    return_full_text=False
)

llm = HuggingFacePipeline(
    pipeline=pipe
)

llm = ChatHuggingFace(
    llm=llm
)

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

In [29]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [30]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [31]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [32]:
ai_message = llm_with_tools.invoke(messages)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [33]:
messages.append(ai_message)

In [34]:
ai_message.tool_calls

[]

In [35]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [36]:
messages.append(
    HumanMessage(
        content="Generate final response using tool output"
    )
)
response = llm_with_tools.invoke(messages).content

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [37]:
response

'As an AI language model, I don\'t have real-time capabilities or access to external tools, but I can walk you through a hypothetical output that a conversion tool might produce. Let\'s assume the tool provided the following output for the conversion of 10 INR to USD given the exchange rate of 1 USD = 83.00 INR:\n\n**Tool Output:**\n"Conversion Result: 10 INR is approximately 0.12 USD."\n\nBased on this hypothetical output from the conversion tool, the final response to your query would be:\n"10 INR is approximately 0.12 USD."'